In [1]:
import pandas as pd
import os
import numpy as np

# Aggregate results of real-world datasets

In [ ]:
dataset = "amazon"
sensitive = "gender"
algorithm = "taucc_fair_max"

root = os.getcwd()
path = root + f"/results/{dataset}/{sensitive}/{algorithm}/init_random"
path

In [ ]:
df = pd.read_csv(path + "/results_runs.csv", sep=";")
df

In [ ]:
if "taucc_fair" in algorithm:
    
    if dataset == "movielens-1m" and sensitive == "age":
        exclude = {'fair_majority', 'fair_minority1', 'fair_minority2', 'run', 'num_iter', 'row_clus', 'col_clus'}
        columns_group = ['fair_majority', 'fair_minority1', 'fair_minority2']
        
    else:
        exclude = {'fair_majority', 'fair_minority', 'run', 'num_iter', 'row_clus', 'col_clus'}
        columns_group = ['fair_majority', 'fair_minority']
        
    metrics = [c for c in df.select_dtypes(include='number').columns if c not in exclude]

    # Aggregazione: mean, std, var per ogni (fair_group1, fair_group2)
    agg_funcs = ['mean', 'std', 'var']

    grouped = (
        df.groupby(columns_group)[metrics]
        .agg(agg_funcs)
    )

    # Flatten del MultiIndex delle colonne: es. tau_x_mean, tau_x_std, ...
    grouped.columns = ['_'.join(col) for col in grouped.columns]
    grouped = grouped.reset_index()

    n_runs = df.groupby(columns_group)['run'].count().values
    if np.all(n_runs == 10):
        print(grouped.shape)   # atteso: (121, n_colonne)
        print(grouped.head())
        #grouped.to_csv(path + "/aggregated.csv", index=False)
    else:
        print("ERROR: run != 10")
        

In [ ]:
if algorithm == "taucc_vanilla":
    
    if len(df) > 10:
        df = df[:10]
    
    exclude = {'run', 'num_iter', 'row_clus', 'col_clus'}
    metrics = [c for c in df.select_dtypes(include='number').columns if c not in exclude]

    agg_funcs = ['mean', 'std', 'var']

    grouped = (
        df.groupby(['run'])[metrics]
        .agg(agg_funcs)
    )

    # Flatten del MultiIndex delle colonne: es. tau_x_mean, tau_x_std, ...
    grouped.columns = ['_'.join(col) for col in grouped.columns]
    grouped = grouped.reset_index()

    #grouped.to_csv(path + "/aggregated.csv", index=False)

# Aggregate results of synthetic data

In [2]:
import pandas as pd
import os
import numpy as np

In [3]:
def get_aggregated(group_keys, exclude_cols, path=None, filename=None, save=False):
    
    exclude_cols = exclude_cols | set(group_keys)
    metrics = [c for c in df.columns if c not in exclude_cols]

    agg_df = (
        df.groupby(group_keys)[metrics]
        .agg(["mean", "std", "var"])
    )

    agg_df.columns = ["_".join(col) for col in agg_df.columns]
    agg_df = agg_df.reset_index()
    
    if save:
        agg_df.to_csv(f"{path}/{filename}.csv", index=False)
    
    return agg_df

In [4]:
clusters = 3
groups = 2
algorithm = "taucc_fair"
rc = True

root = os.getcwd()

if rc:
    path = root + f"/results/synthetic/clus{clusters}_rc/{algorithm}"
else:
    path = root + f"/results/synthetic/clus{clusters}/{algorithm}"

exclude_cols = {"run", "num_iter", "row_clus", "col_clus"}

if rc:
    group_keys = ["sensitive_px", "sensitive_py"]
else:
    group_keys = ["sensitive_p"]

if algorithm == "taucc_vanilla":
    separator = ";"
    
elif algorithm == "taucc_fair_max" or algorithm == "taucc_fair":
    separator = ","
    
    if groups == 2:
        if rc:
            group_keys.extend(["row_fair_majority", "row_fair_minority", "col_fair_majority", "col_fair_minority"])
        else:
            group_keys.extend(["fair_majority", "fair_minority"])
    else:
        group_keys.extend(["fair_majority", "fair_minority1", "fair_minority2"])
else:
    raise Exception("Exception")

df = pd.read_csv(path + f"/results_runs_groups{groups}.csv", sep=separator)
df.drop("num_groups", axis=1, inplace=True)
filename = f"aggregated_groups{groups}"

In [5]:
if rc:
    df['balance_bera'] = df[['balance_bera_rows', 'balance_bera_cols']].min(axis=1)
    df = df.drop(columns=['balance_bera_rows', 'balance_bera_cols'])
#df

In [6]:
group_keys

['sensitive_px',
 'sensitive_py',
 'row_fair_majority',
 'row_fair_minority',
 'col_fair_majority',
 'col_fair_minority']

In [7]:
path

'/home/peiretti/fair-clustering/results/synthetic/clus3_rc/taucc_fair'

In [8]:
get_aggregated(group_keys, exclude_cols, path, filename, save=True)

,sensitive_px,sensitive_py,row_fair_majority,row_fair_minority,col_fair_majority,col_fair_minority,tau_x_mean,tau_x_std,tau_x_var,tau_y_mean,...,AMI_van_cols_var,ARI_van_cols_mean,ARI_van_cols_std,ARI_van_cols_var,time_mean,time_std,time_var,balance_bera_mean,balance_bera_std,balance_bera_var
0,0.8,0.8,0.0,0.0,0.0,0.00,0.395020,0.000000,0.000000,0.395020,...,0.000000e+00,1.000000,0.000000,0.000000e+00,0.056589,0.026863,0.000722,0.514226,0.000000,0.000000
1,0.8,0.8,0.0,0.0,0.0,0.25,0.395020,0.000000,0.000000,0.395020,...,0.000000e+00,1.000000,0.000000,0.000000e+00,0.057276,0.016044,0.000257,0.514226,0.000000,0.000000
2,0.8,0.8,0.0,0.0,0.0,0.50,0.382949,0.015778,0.000249,0.197831,...,1.457235e-03,0.609876,0.020913,4.373605e-04,0.957748,0.916296,0.839598,0.000000,0.000000,0.000000
3,0.8,0.8,0.0,0.0,0.0,0.75,0.370757,0.027174,0.000738,0.129064,...,2.311815e-03,0.412701,0.033671,1.133765e-03,0.220355,0.057078,0.003258,0.000000,0.000000,0.000000
4,0.8,0.8,0.0,0.0,0.0,1.00,0.350700,0.042716,0.001825,0.091443,...,2.654552e-03,0.282777,0.041933,1.758417e-03,0.199267,0.058889,0.003468,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
620,0.8,0.8,1.0,1.0,1.0,0.00,0.042857,0.002378,0.000006,0.113689,...,3.126445e-05,0.404558,0.003981,1.584781e-05,27.964327,0.904948,0.818931,0.000000,0.000000,0.000000
621,0.8,0.8,1.0,1.0,1.0,0.25,0.039937,0.003981,0.000016,0.094530,...,5.011261e-06,0.331235,0.001491,2.222665e-06,26.074785,1.567927,2.458394,0.250114,0.000000,0.000000
622,0.8,0.8,1.0,1.0,1.0,0.50,0.040274,0.003803,0.000014,0.082730,...,0.000000e+00,0.250392,0.000000,0.000000e+00,26.036772,2.207029,4.870976,0.304780,0.000000,0.000000
623,0.8,0.8,1.0,1.0,1.0,0.75,0.032979,0.003640,0.000013,0.043640,...,1.984235e-08,0.149571,0.000120,1.449843e-08,24.192374,3.203602,10.263063,0.464160,0.003415,0.000012


# Best run Vanilla (synthetic data)

In [ ]:
import pandas as pd
import numpy as np

def best_run(group):
    max_tx = group["tau_x"].max()
    max_ty = group["tau_y"].max()
    dist = np.sqrt((group["tau_x"] - max_tx) ** 2 + (group["tau_y"] - max_ty) ** 2)
    return group.loc[dist.idxmin()]

clusters = 3
groups = 2

root = os.getcwd()
path = root + f"/results/synthetic/clus{clusters}/taucc_vanilla"

df = pd.read_csv(path + f"/results_runs_groups{groups}.csv", sep=";")

best_runs = df.groupby("sensitive_p").apply(best_run).reset_index(drop=True)
int_cols = ["num_groups", "run", "num_iter", "row_clus", "col_clus"]
best_runs[int_cols] = best_runs[int_cols].astype(int)
best_runs.to_csv(path + f"/best_run_groups{groups}.csv", index=False)